# MATH840 — Lab 8: Combinations, neural networks and foundation models

**Weeks 9–10 | graded assignment 7 of 7 | 8 points | due Sunday 1 November 2026, 23:59**

Your series from Week 4, one last time. You have an ETS model (Lab 5) and a seasonal ARIMA (Lab 7), each
with a score on the hidden horizon. Those are the models to beat. Three ways to try, from the last two
weeks of lectures:

| Track | The idea | Lecture |
|---|---|---|
| **A. Combine** | average the models you already have | *Advanced Forecasting Methods* |
| **B. Train** | a neural network — MLP, N-BEATS, LSTM — trained on your series | *Neural Networks for Forecasting* |
| **C. Borrow** | a foundation model — Chronos-2, TimesFM 2.5, TiRex — that has never seen your series | *Foundation Models for Time Series* |
| **D. Your own way** | any mix of the above, all three, or something else you can defend | — |

**You choose.** One track done well is a full-marks lab. Doing more is welcome and counts toward the
bonus. Whatever you take, everything is judged the same way: the same four rolling windows, the same
benchmark, the same MASE.

No plots of the series, no decomposition, no ACF: you have read your series three times.

**Accuracy is marked**, as in Labs 5 and 7: beat your benchmark — 3 points, within 10% — 2, worse — 1
with a written explanation.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | Your models, rebuilt — the ones to beat | 1 |
| 2 | Forecast accuracy against your benchmark (scored from your file) | 3 |
| 3 | Justification block: your track, the interval, conclusions against the whole course | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab09_forecast.csv`, `SURNAME_lab09.pdf` and `SURNAME_lab09.ipynb` to the Lab 8
activity on Moodle by **Sunday 1 November 2026, 23:59**.

## 0. Setup

Given — run it and move on. Nothing in this section is marked. Tracks B and C need extra packages
(installed here) and, for C, about 1.3 GB of model weights, downloaded only when you first call a
foundation model. A CPU runtime is enough for every track.

In [ ]:
!pip install -q statsforecast==2.1.1 neuralforecast==3.3.0 chronos-forecasting==2.3.2 "timesfm[torch]==3.0.2" tirex-ts==1.4.2

import hashlib
import logging
import os
import time
import warnings
warnings.filterwarnings("ignore")
os.environ["NIXTLA_ID_AS_COL"] = "true"
for name in ("lightning", "lightning.pytorch", "pytorch_lightning", "transformers"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, MSTL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)
print("running on", DEVICE)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier produces the same series as in Weeks 4, 6, 7 and 8. If the code printed below is not
the one from your earlier submissions, stop and check `MY_ID`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)
dates = s["ds"]

print(f"series {CODE}: {len(s)} observations, {dates.min().date()} to {dates.max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")

### Helpers: the benchmark and your models

Given. The benchmark, `mase` and the validation split are the same as in every lab since Week 4.
`fit_ets` is Lab 5's, `fit_arima` is Lab 7's, `fit_mstl` is new; all three take `log=True` and then return
forecasts and intervals on the original scale.

In [ ]:
def benchmark_forecasts(history: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    # The four simple methods from Week 1.
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


train, valid = y[:-H], y[-H:]
simple = benchmark_forecasts(train, H, M)
BENCHMARK = min(simple, key=lambda name: rmse(valid, simple[name]))
print(f"your benchmark: {BENCHMARK} (validation MASE {mase(valid, simple[BENCHMARK]):.3f})")

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())


class Fitted:
    # A fitted model whose forecasts are on the original scale, whatever it was fitted on.
    def __init__(self, model, log):
        self.model, self.log = model, log

    def predict(self, h: int, level=None):
        out = self.model.predict(h=h, level=level)
        return {k: np.exp(v) for k, v in out.items()} if self.log else out


def fit_ets(history, spec=None, damped=None, log=False) -> Fitted:
    # Lab 5: spec=None lets AutoETS search; "MAM", "AAdA" (damped) and so on fix the model.
    x = np.log(history) if log else np.asarray(history, float)
    if spec is None:
        return Fitted(AutoETS(season_length=M).fit(y=x), log)
    spec = str(spec).strip().upper()
    if spec[2:4] in ("DA", "DM", "DN"):
        spec, damped = spec[:2] + spec[3:], True
    return Fitted(AutoETS(season_length=M, model=spec,
                          damped=bool(damped) if spec[1] != "N" else None).fit(y=x), log)


def fit_arima(history, order=None, seasonal=(0, 0, 0), log=False, drift=False) -> Fitted:
    # Lab 7: order=(p, d, q) and seasonal=(P, D, Q) for a hand specification; order=None for AutoARIMA.
    x = np.log(history) if log else np.asarray(history, float)
    if order is None:
        model = AutoARIMA(season_length=M)
    else:
        model = ARIMA(order=tuple(order), seasonal_order=tuple(seasonal), season_length=M,
                      include_drift=drift and (order[1] + seasonal[1]) == 1)
    return Fitted(model.fit(y=x), log)


def fit_mstl(history, log=False) -> Fitted:
    # New: STL with the seasonal period M, an automatic ETS without a season for the rest.
    x = np.log(history) if log else np.asarray(history, float)
    return Fitted(MSTL(season_length=M).fit(y=x), log)


FITTERS = {"ets": fit_ets, "arima": fit_arima, "mstl": fit_mstl}


def fit_member(member, history) -> Fitted:
    kind, args = member
    return FITTERS[kind](history, **args)

### Helpers: forecasters

A **forecaster** is a function `f(history, h)` that returns a dictionary with the point forecast `"mean"`
and the 80% bounds `"lo-80"`, `"hi-80"` (NaN when the method has no interval of its own). Three makers,
one per family:

- `classic(member)` — a Lab 5 / Lab 7 / MSTL model (track A);
- `neural(MLP)`, `neural(NBEATS)`, `neural(LSTM)` — a network trained **on `history` only**, from scratch,
  every time it is called (track B);
- `foundation("Chronos-2")`, `foundation("TimesFM 2.5")`, `foundation("TiRex")` — zero-shot, the 10% and
  90% quantiles as the interval; the weights are downloaded on first use (track C).

Writing your own forecaster for track D is the same: any function `f(history, h)` returning that
dictionary.

In [ ]:
NAN = lambda h: np.full(h, np.nan)
_pretrained = {}


def classic(member):
    def forecast(history, h):
        return fit_member(member, history).predict(h=h, level=[80])
    return forecast


def neural(model_class, input_size=None, max_steps=300, seed=1, **settings):
    # Trained on the history it is given and nothing else. input_size: how many past steps it reads.
    from neuralforecast import NeuralForecast

    def forecast(history, h):
        df = pd.DataFrame({"unique_id": CODE, "ds": dates.iloc[:len(history)].to_numpy(), "y": history})
        model = model_class(h=h, input_size=input_size or 3 * M, max_steps=max_steps, scaler_type="robust",
                            random_seed=seed, accelerator="gpu" if DEVICE == "cuda" else "cpu", devices=1,
                            enable_progress_bar=False, enable_model_summary=False, logger=False, **settings)
        nf = NeuralForecast(models=[model], freq=FREQ)
        nf.fit(df=df)
        return {"mean": nf.predict()[model_class.__name__].to_numpy(), "lo-80": NAN(h), "hi-80": NAN(h)}
    return forecast


def pretrained(name):
    # Loads a foundation model once, the first time it is asked for.
    if name not in _pretrained:
        if name == "Chronos-2":
            from chronos import BaseChronosPipeline
            _pretrained[name] = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
        elif name == "TimesFM 2.5":
            import timesfm
            model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
            model.compile(timesfm.ForecastConfig(max_context=512, max_horizon=128, normalize_inputs=True,
                                                 use_continuous_quantile_head=True, fix_quantile_crossing=True))
            _pretrained[name] = model
        elif name == "TiRex":
            import tirex
            _pretrained[name] = tirex.load_model("NX-AI/TiRex", device=DEVICE)
        else:
            raise ValueError(name)
    return _pretrained[name]


def foundation(name):
    def forecast(history, h):
        model = pretrained(name)
        context = torch.tensor(np.asarray(history, float), dtype=torch.float32)
        if name == "Chronos-2":
            q, _ = model.predict_quantiles([context], prediction_length=h, quantile_levels=[0.1, 0.5, 0.9])
            q = np.asarray(q[0]).reshape(-1, h, 3)[0]
            return {"mean": q[:, 1], "lo-80": q[:, 0], "hi-80": q[:, 2]}
        if name == "TimesFM 2.5":
            _, q = model.forecast(horizon=h, inputs=[context.numpy()])
            q = np.asarray(q)[0, :h]                       # mean, then the 10%, 20%, ..., 90% quantiles
            return {"mean": q[:, 5], "lo-80": q[:, 1], "hi-80": q[:, 9]}
        q, _ = model.forecast(context=[context], prediction_length=h)
        q = np.asarray(q)[0]                               # TiRex: the 10%, 20%, ..., 90% quantiles
        return {"mean": q[:, 4], "lo-80": q[:, 0], "hi-80": q[:, 8]}
    return forecast

### Helpers: one protocol for every track

- `backtest` runs forecasters on four rolling origins, $h$ steps each (window 1 is the validation window
  of every lab since Week 4), and records how long each took.
- `add` puts a new backtest next to the ones you already have, so every track lands in one table, `bt`.
- `combine` adds an equal-weight average of two or more columns of `bt`.
- `window_table` reads `bt`: MASE per window, the mean, windows won, seconds.
- `interval_check` judges the 80% interval recipes on windows they did not see.
- `final_forecast` produces your submission from the **whole** history for any column of `bt`.

In [ ]:
WINDOWS = 4
FORECASTERS = {}      # every forecaster you have backtested, by name
MIXES = {}            # every average you have built with combine: its parts


def backtest(forecasters: dict, history: np.ndarray = y, h: int = H, windows: int = WINDOWS) -> pd.DataFrame:
    FORECASTERS.update(forecasters)
    frames, seconds = [], {}
    for k in range(windows, 0, -1):
        end = len(history) - (k - 1) * h
        past, actual = history[:end - h], history[end - h:end]
        frame = pd.DataFrame({"window": k, "step": np.arange(1, h + 1), "actual": actual,
                              "benchmark": benchmark_forecasts(past, h, M)[BENCHMARK]})
        for name, f in forecasters.items():
            start = time.time()
            fc = f(past, h)
            seconds[name] = seconds.get(name, 0.0) + time.time() - start
            frame[name], frame[f"{name} lo"], frame[f"{name} hi"] = fc["mean"], fc["lo-80"], fc["hi-80"]
        frames.append(frame)
    bt = pd.concat(frames, ignore_index=True)
    bt.attrs["seconds"] = seconds
    return bt


def add(bt: pd.DataFrame, new: pd.DataFrame) -> pd.DataFrame:
    # The same windows side by side: new columns replace old ones with the same name.
    out = bt.drop(columns=[c for c in new.columns if c in bt.columns and c not in ("window", "step", "actual", "benchmark")])
    cols = [c for c in new.columns if c not in out.columns]
    out = pd.concat([out, new[cols]], axis=1)
    out.attrs["seconds"] = {**bt.attrs.get("seconds", {}), **new.attrs.get("seconds", {})}
    return out


def combine(bt: pd.DataFrame, names: list[str], label: str) -> None:
    # Equal-weight average of the forecasts, and the average of their bounds (NaN if any part has none).
    MIXES[label] = list(names)
    bt[label] = bt[names].mean(axis=1)
    bt[f"{label} lo"] = bt[[f"{n} lo" for n in names]].mean(axis=1, skipna=False)
    bt[f"{label} hi"] = bt[[f"{n} hi" for n in names]].mean(axis=1, skipna=False)


def forecasts_in(bt: pd.DataFrame) -> list[str]:
    # Every forecast column of bt, the benchmark last.
    names = [c for c in bt.columns if c not in ("window", "step", "actual", "benchmark")
             and not c.endswith((" lo", " hi"))]
    return names + ["benchmark"]


def window_table(bt: pd.DataFrame, columns: list[str] = None) -> pd.DataFrame:
    # MASE of each forecast in each window, the mean over windows, how often it was best, and its cost.
    columns = columns or forecasts_in(bt)
    table = bt.groupby("window").apply(
        lambda g: pd.Series({c: mase(g["actual"], g[c]) for c in columns}), include_groups=False).T
    table = table[sorted(table.columns, reverse=True)]
    table.columns = [f"window {w}" + (" (validation)" if w == 1 else "") for w in table.columns]
    table["mean"] = table.mean(axis=1)
    best = bt.groupby("window").apply(
        lambda g: min(columns, key=lambda c: mase(g["actual"], g[c])), include_groups=False)
    table["windows won"] = [int((best == c).sum()) for c in columns]
    table["seconds"] = [bt.attrs.get("seconds", {}).get(c, np.nan) for c in columns]
    return table


def winkler(actual, lo, hi, alpha=0.2) -> float:
    # Width of the interval plus 2/alpha times every miss (lecture, Week 8), in units of SCALE.
    actual, lo, hi = map(np.asarray, (actual, lo, hi))
    return float(np.mean((hi - lo) + 2 / alpha * (np.maximum(lo - actual, 0) + np.maximum(actual - hi, 0))) / SCALE)


def conformal_radius(errors) -> float:
    # The 80th percentile of the absolute errors: the forecast plus or minus this is the 80% interval.
    return float(np.quantile(np.abs(np.asarray(errors)), 0.80))


def interval_check(bt: pd.DataFrame, label: str) -> pd.DataFrame:
    # The forecast's own interval (if it has one: a model's own, or the averaged bounds of a mix) and the
    # conformal one, each judged on windows it did not see: the conformal radius for a window comes from
    # the errors of the OTHER windows only.
    rows = []
    for w, g in bt.groupby("window"):
        others = bt[bt["window"] != w]
        r = conformal_radius(others["actual"] - others[label])
        recipes = [("conformal", g[label] - r, g[label] + r)]
        if g[f"{label} lo"].notna().all():
            recipes.insert(0, ("own interval", g[f"{label} lo"], g[f"{label} hi"]))
        for recipe, lo, hi in recipes:
            rows.append({"recipe": recipe, "window": w,
                         "coverage": float(np.mean((g["actual"] >= lo) & (g["actual"] <= hi))),
                         "Winkler": winkler(g["actual"], lo, hi), "width": float(np.mean(hi - lo)) / SCALE})
    return pd.DataFrame(rows).groupby("recipe")[["coverage", "Winkler", "width"]].mean()


def _from_all_of_y(label: str) -> dict:
    # Calls every forecaster inside `label` again on the whole history y and averages as combine did.
    if label in MIXES:
        parts = [_from_all_of_y(n) for n in MIXES[label]]
        return {k: np.mean([p[k] for p in parts], axis=0) for k in ("mean", "lo-80", "hi-80")}
    return {k: np.asarray(v, float) for k, v in FORECASTERS[label](y, H).items() if k in ("mean", "lo-80", "hi-80")}


def final_forecast(bt: pd.DataFrame, label: str, interval: str):
    # Your submission: the point forecast of `label` from all of y, and the interval recipe you chose.
    fc = _from_all_of_y(label)
    if interval == "conformal":
        r = conformal_radius(bt["actual"] - bt[label])      # the radius from all four windows
        return fc["mean"], fc["mean"] - r, fc["mean"] + r
    if interval == "own interval":
        if np.isnan(fc["lo-80"]).any():
            raise ValueError(f"{label} has no interval of its own: use 'conformal'")
        return fc["mean"], fc["lo-80"], fc["hi-80"]
    raise ValueError("interval is 'own interval' or 'conformal'")

## 1. Your models, rebuilt — the ones to beat

*1 point.* Copy your **final** models from Lab 5 and Lab 7 — the ones you submitted — into `MEMBERS`, and
add MSTL. Then copy the accuracy scores you were sent after Labs 5 and 7 into `PAST`. Everyone does this
section, whatever the track: these are the models every new idea has to beat.

A member is `(kind, arguments)`:

- Lab 5: `("ets", {"spec": "MAM"})`, `("ets", {"spec": "AAdA"})` or `("ets", {"spec": None})` for `AutoETS`;
- Lab 7: `("arima", {"order": (0, 1, 1), "seasonal": (0, 1, 1), "log": True})` or
  `("arima", {"order": None, "log": False})` for `AutoARIMA`;
- MSTL: `("mstl", {"log": ...})` — the log decision is yours, the same as in Lab 7 or argued anew.

Did not submit Lab 5 or Lab 7? Use the automatic model and say so; it costs nothing.

In [ ]:
MEMBERS = {
    "ETS (Lab 5)":    ("ets", {"spec": None}),                     # <-- your Lab 5 submission
    "ARIMA (Lab 7)":  ("arima", {"order": None, "log": False}),     # <-- your Lab 7 submission
    "MSTL":           ("mstl", {"log": False}),                     # <-- log or not
}

PAST = {                                     # <-- the scores you were sent: MASE and skill on the hidden horizon
    "Lab 5 (ETS)":            {"hidden MASE": None, "skill": None},
    "Lab 7 (seasonal ARIMA)": {"hidden MASE": None, "skill": None},
}

**Check that the rebuild is right.** The validation-window MASE below must match what your Lab 5 and Lab 7
notebooks printed for the same models (`valid MASE`). If a number differs, find out why before going on —
and say so below.

In [ ]:
rebuilt = pd.Series({name: mase(valid, fit_member(member, train).predict(h=H)["mean"])
                     for name, member in MEMBERS.items()}, name="validation MASE")
rebuilt["benchmark"] = mase(valid, simple[BENCHMARK])
rebuilt.to_frame()

Now the same models on all four rolling windows. Run once — it refits every member four times (a few
minutes for a monthly ARIMA). This `bt` is the table every track adds to.

In [ ]:
bt = backtest({name: classic(member) for name, member in MEMBERS.items()})
window_table(bt)

**Do the rebuilt numbers match your earlier notebooks? Where does MSTL land next to your two models, and
where do they all stand against the benchmark across the windows?**

→

## 2. Your track, and your forecast

*3 points, scored from your submission file* on the hidden horizon against your benchmark:
skill = your MASE / your benchmark's MASE; below 1 — 3 points, up to 1.1 — 2, above — 1.

**What you may submit:** any column of your final `bt` — one forecaster, or an average you built with
`combine` — with an 80% interval: its own, or conformal. Chosen on the rolling windows, never on anything
else.

⚠️ **One hard rule: train only on your own series.** Several Track B series are cut from the same source;
another student's history can contain your hidden horizon. A network trained on the other series would be
trained on your answer.

### 2.0 Your plan, before any results

Which track or tracks you take, why, and what you expect to see. Written now, before 2.1–2.4 — it is
the "before" half of your conclusions.

**My track(s), and what I expect.**

→

### 2.1 Track A — combine your models

*Skip if you do not take track A.* The *Advanced Forecasting Methods* lecture and the Week 9 practice.

**Before you look at the table**, write down which members go into your combination and why — each must
be defensible on its own (a combination is as good as the worst member it cannot outvote).

**My combination, decided before the table, and why.**

→

In [ ]:
COMBINATION = list(MEMBERS)               # <-- the members you average: two or three of MEMBERS
combine(bt, list(MEMBERS), "average of all")
if COMBINATION != list(MEMBERS):
    combine(bt, COMBINATION, "my combination")

table = window_table(bt)
print(f"selection on the rolling mean would have chosen: {table.loc[list(MEMBERS), 'mean'].idxmin()}")
table

In [ ]:
# Do the members make different mistakes? A combination gains most when their errors are not correlated.
errors = pd.DataFrame({n: bt["actual"] - bt[n] for n in MEMBERS})
print("correlation of the members' errors over all windows:")
errors.corr().round(2)

### 2.2 Track B — train a neural network on your series

*Skip if you do not take track B.* The *Neural Networks for Forecasting* lecture. The settings below are
a reasonable start: the network reads the last `3 * M` observations and trains for 300 steps. You may
change `input_size`, `max_steps` or `seed` — say what and why in 3.1. Three networks on four windows: a
few minutes on a CPU.

In [ ]:
from neuralforecast.models import LSTM, MLP, NBEATS

NEURAL = {
    "MLP":     neural(MLP),
    "N-BEATS": neural(NBEATS),
    "LSTM":    neural(LSTM),
}
bt = add(bt, backtest(NEURAL))
window_table(bt, list(MEMBERS) + list(NEURAL) + ["benchmark"])

In [ ]:
# TODO (track B): your best network once more with another seed, e.g.
#   bt = add(bt, backtest({"N-BEATS seed 2": neural(NBEATS, seed=2)}))
# — how much of its score is the network, and how much the luck of its starting weights?

### 2.3 Track C — borrow a foundation model

*Skip if you do not take track C.* The *Foundation Models for Time Series* lecture and the Week 10
practice. No training: each model reads the history of the window and forecasts, with its own 80%
interval (the 10% and 90% quantiles). The first call downloads the weights.

In [ ]:
FOUNDATION = {name: foundation(name) for name in ["Chronos-2", "TimesFM 2.5", "TiRex"]}
bt = add(bt, backtest(FOUNDATION))
window_table(bt, list(MEMBERS) + list(FOUNDATION) + ["benchmark"])

### 2.4 Track D — your own way

*Optional.* Anything you can defend, on the same windows: every track at once, an average of old and new
decided in advance (`combine(bt, ["ETS (Lab 5)", "ARIMA (Lab 7)", "Chronos-2"], "old + new")`), tuned
networks, more seeds, a shorter history, Chronos-Bolt, TimeGPT (key from a Colab secret, never in the
notebook) — or a forecaster of your own, any `f(history, h)` passed to `backtest`.

In [ ]:
# Your own forecasters and mixes, if any.

### 2.5 The scoreboard

Everything you have run, on the same four windows.

In [ ]:
scoreboard = window_table(bt)
scoreboard.sort_values("mean")

In [ ]:
# TODO: one plot of your choice that compares what you tried across the windows
# (for example: mean MASE as bars, or MASE by window as lines, one per forecaster).

### 2.6 Your choice and its interval

`CHOICE` is one row of the scoreboard. `interval_check` judges its interval recipes on windows they did
not see.

In [ ]:
CHOICE = "average of all"                    # <-- a row of the scoreboard
interval_check(bt, CHOICE)

### 2.7 The submission

`final_forecast` calls every forecaster inside your choice once more on the **whole** history `y` (the
networks train again) and averages them as `combine` did. For the conformal interval the radius comes from
**all four** windows of `bt`.

In [ ]:
INTERVAL = "conformal"                       # <-- "own interval" or "conformal"
YHAT, YHAT_LO_80, YHAT_HI_80 = final_forecast(bt, CHOICE, INTERVAL)

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H or np.isnan(np.ravel(value)).any():
        raise ValueError(f"{name} must be an array of length H = {H} with no NaN")
pd.DataFrame({"ds": FUTURE.date, "yhat": YHAT, "lo 80": YHAT_LO_80, "hi 80": YHAT_HI_80})

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own cells,
not adjectives.

**3.1 Your track** *(1)* — what you tried and what the windows show, against your Section 1 models and the
benchmark. Answer the questions of your track; if you took several, a short paragraph for each.

- *A. Combine:* the members you chose before the table and why; who won how many windows and where the
  combination landed; what the error correlations say about how much there was to gain; what selection
  would have submitted, and why you combined anyway — or did not.
- *B. Train:* how MLP, N-BEATS and LSTM did window by window, and why — how many observations they
  learned from, what `input_size` means for your series, what the second seed changed; what they cost in
  seconds; anything you changed.
- *C. Borrow:* how Chronos-2, TimesFM 2.5 and TiRex did window by window and what they cost. Your series
  is disguised public data: can you rule out that a model saw its source in pretraining? What would you
  need to know, and how does it change your trust in the numbers?
- *D. Your own way:* what question your extra work answered, and what the answer was — including "nothing".

→

**3.2 The interval** *(1)* — the recipes for your choice on unseen windows: coverage, Winkler score,
width. The one you submitted and why. How many errors the conformal radius rests on, and what that means
for trusting it.

→

**3.3 What you submit, against the whole course** *(1)* — your Lab 5 and Lab 7 hidden-horizon skills
(`PAST`) next to tonight's rolling evidence. What you submitted and why; the skill you expect, and why;
how it compares with what you wrote in 2.0. If you had to forecast this series at work every month,
which approach would you use, and what would you check — one paragraph.

→

In [ ]:
# Optional, for 3.3: your past results.
pd.DataFrame(PAST).T

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: everything is compared on the rolling windows; only the final forecast sees all of `y`; no
  network is trained on any series but yours.
- Every number in your text comes from a cell.
- No API key in the notebook.

## 5. Submit

Run both cells, then upload **three files** to the Lab 8 activity on Moodle by Sunday 1 November 2026,
23:59:

- `SURNAME_lab09_forecast.csv` — written below;
- `SURNAME_lab09.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab09.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab09_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")